# 第6课：模拟Basis并计算Harvest Cash Price

本课接着第5课的10,000个共同情景，加入Iowa harvest basis，并计算农民收获时真正面对的cash price。

本课仍然**不计算套保合约、期货P&L或最终利润**。请从上到下逐格运行。

## 0. 先把最重要的概念讲直白

期货价格是交易所里的价格，但农民把玉米卖给当地粮库时，拿到的是当地现金价格。两者通常不完全相同。

本项目采用Iowa State的定义：

$$Basis = CashPrice - FuturesPrice$$

所以：

$$CashPrice = FuturesPrice + Basis$$

例如Harvest futures是$4.50/bu，而basis是−$0.20/bu，则当地cash price为：

$$4.50 + (-0.20) = 4.30\text{ dollars per bushel}$$

## 1. 本课数据来源与三角分布

来源：Iowa State University Ag Decision Maker, **Iowa Corn Price Basis, File A2-41**，州平均、November week 1、December futures contract，2020/21–2024/25。

原表给出：

| 信息 | 原表数值 | 模型中的角色 |
|---|---:|---|
| 五年最负的basis | −0.28 | 数值下限 $a$ |
| 五年平均basis | −0.20 | 最可能值/众数 $c$ |
| 五年最不负的basis | −0.11 | 数值上限 $b$ |

因此：

$$Basis_i \sim Triangular(a=-0.28,c=-0.20,b=-0.11)$$

注意：原表文字把“maximum basis”定义为最负、把“minimum basis”定义为最不负。为避免混乱，我们在代码中按**数字大小**命名为`BASIS_LOW=-0.28`和`BASIS_HIGH=-0.11`。

## 2. 导入工具并锁定设置

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
N_HISTORICAL_YEARS = 30
RANDOM_SEED = 8_122_026

BASIS_LOW = -0.28
BASIS_MODE = -0.20
BASIS_HIGH = -0.11
CASH_PRICE_FLOOR = 0.50

print('模拟次数:', N_SIMULATIONS)
print('Basis三角分布:', (BASIS_LOW, BASIS_MODE, BASIS_HIGH))
print('Cash price下限假设: $', CASH_PRICE_FLOOR, '/bu')

## 3. 读取第5课输出

先运行第5课，系统会生成：

`lesson_05_outputs/futures_scenarios_10000.csv`

本课直接读取它，这样不会重复前面30年的回归与Residual计算。

In [ ]:
candidate_paths = [
    Path.cwd() / 'lesson_05_outputs' / 'futures_scenarios_10000.csv',
    Path.cwd().parent / 'lesson_05_outputs' / 'futures_scenarios_10000.csv',
]

lesson5_path = next((p for p in candidate_paths if p.exists()), None)

if lesson5_path is None:
    raise FileNotFoundError(
        '没有找到第5课CSV。请先运行Lesson_05 Notebook的全部单元格，'
        '并确保lesson_05_outputs文件夹与Notebook位于同一工作目录。'
    )

scenarios = pd.read_csv(lesson5_path)
print('读取文件:', lesson5_path)
print('行数:', len(scenarios))
print('列数:', len(scenarios.columns))

## 4. 先检查第5课输入是否完整

本课最关键的输入是`harvest_futures_usd_per_bushel`。同时保留产量、天气和来源年份，后面所有策略才能继续使用同一组情景。

In [ ]:
required_columns = [
    'scenario_id',
    'weather_source_year',
    'yield_residual_source_year',
    'price_residual_source_year',
    'july_pdsi',
    'july_yield_forecast_bu_per_acre',
    'final_yield_bu_per_acre',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
]

missing = [c for c in required_columns if c not in scenarios.columns]
assert not missing, f'缺少列: {missing}'
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[required_columns].isna().sum().sum() == 0

print('第5课输入检查通过。')
print(scenarios[required_columns].head(5).round(4).to_string(index=False))

## 5. 为什么先抽Uniform(0,1)？

三角分布可以从一个均匀随机数 $U_i\sim Uniform(0,1)$ 转换出来。

分界点为：

$$p=\frac{c-a}{b-a}$$

当 $U_i<p$：

$$Basis_i=a+\sqrt{U_i(b-a)(c-a)}$$

当 $U_i\ge p$：

$$Basis_i=b-\sqrt{(1-U_i)(b-a)(b-c)}$$

把公式写出来，比只调用一个黑箱随机函数更适合课程作业，也方便之后在Excel中逐行复现。

In [ ]:
basis_cutoff = (BASIS_MODE - BASIS_LOW) / (BASIS_HIGH - BASIS_LOW)
theoretical_basis_mean = (BASIS_LOW + BASIS_MODE + BASIS_HIGH) / 3

print(f'三角分布分界点 p = {basis_cutoff:.6f}')
print(f'理论平均basis = {theoretical_basis_mean:.6f} $/bu')

## 6. 延续第5课的同一条随机数序列

第5课依次抽取了：天气年份、产量Residual年份、成对价格Residual年份。

为了让本课Basis抽样与整个正式模型完全一致，我们用相同seed重新建立随机数生成器，先消耗前三组整数抽样，再取得10,000个basis uniform draws。

这一步只是在恢复第5课结束时的随机数位置，不会改变已经生成的期货价格。

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# 与第5课完全相同的前三次抽样；这里只移动随机数状态
_ = rng.integers(0, N_HISTORICAL_YEARS, size=N_SIMULATIONS)  # weather rows
_ = rng.integers(0, N_HISTORICAL_YEARS, size=N_SIMULATIONS)  # yield residual rows
_ = rng.integers(0, N_HISTORICAL_YEARS, size=N_SIMULATIONS)  # paired price rows

basis_uniform_draw = rng.random(N_SIMULATIONS)

print('前5个basis uniform draws:', basis_uniform_draw[:5])
print(f'Uniform样本平均值 = {basis_uniform_draw.mean():.6f}')

## 7. 用Inverse CDF公式生成Basis

In [ ]:
basis = np.where(
    basis_uniform_draw < basis_cutoff,
    BASIS_LOW + np.sqrt(
        basis_uniform_draw
        * (BASIS_HIGH - BASIS_LOW)
        * (BASIS_MODE - BASIS_LOW)
    ),
    BASIS_HIGH - np.sqrt(
        (1.0 - basis_uniform_draw)
        * (BASIS_HIGH - BASIS_LOW)
        * (BASIS_HIGH - BASIS_MODE)
    ),
)

print('前10个模拟basis:')
print(pd.Series(basis[:10], name='basis_usd_per_bushel').round(6).to_string(index=False))

## 8. 计算Harvest Cash Price

核心公式只有一个：

$$CashPrice_i=HarvestFutures_i+Basis_i$$

代码中的$0.50/bu下限是显式模型假设，用来防止极端模拟产生不合理的负现金价格。本次情景中会检查它是否真正触发。

In [ ]:
harvest_futures = scenarios['harvest_futures_usd_per_bushel'].to_numpy()
raw_cash_price = harvest_futures + basis
cash_price = np.maximum(CASH_PRICE_FLOOR, raw_cash_price)

scenarios['basis_uniform_draw'] = basis_uniform_draw
scenarios['basis_usd_per_bushel'] = basis
scenarios['cash_price_usd_per_bushel'] = cash_price

print(scenarios[[
    'scenario_id',
    'harvest_futures_usd_per_bushel',
    'basis_usd_per_bushel',
    'cash_price_usd_per_bushel',
]].head(10).round(6).to_string(index=False))

## 9. 用一行情景手算验证

我们拿第1行情景检查：`harvest futures + basis`是否真的等于`cash price`。

In [ ]:
row1 = scenarios.iloc[0]
manual_cash_price = (
    row1['harvest_futures_usd_per_bushel']
    + row1['basis_usd_per_bushel']
)

print(f"Scenario 1 Harvest Futures = {row1['harvest_futures_usd_per_bushel']:.6f}")
print(f"Scenario 1 Basis = {row1['basis_usd_per_bushel']:.6f}")
print(f'Manual Cash Price = {manual_cash_price:.6f}')
print(f"Stored Cash Price = {row1['cash_price_usd_per_bushel']:.6f}")

assert np.isclose(manual_cash_price, row1['cash_price_usd_per_bushel'])
print('手算验证通过。')

## 10. 必须通过的模型检查

In [ ]:
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[['basis_uniform_draw', 'basis_usd_per_bushel', 'cash_price_usd_per_bushel']].isna().sum().sum() == 0
assert ((basis_uniform_draw >= 0) & (basis_uniform_draw < 1)).all()
assert (basis >= BASIS_LOW - 1e-12).all()
assert (basis <= BASIS_HIGH + 1e-12).all()
assert (cash_price >= CASH_PRICE_FLOOR).all()
assert np.allclose(cash_price, np.maximum(CASH_PRICE_FLOOR, harvest_futures + basis))

basis_floor_count = int((basis <= BASIS_LOW + 1e-12).sum())
cash_floor_count = int((cash_price <= CASH_PRICE_FLOOR + 1e-12).sum())

print('全部模型检查通过。')
print('Basis恰好碰到数值下限的次数:', basis_floor_count)
print('Cash price碰到$0.50下限的次数:', cash_floor_count)

## 11. 汇总Basis与Cash Price分布

均值告诉我们中心位置，标准差和分位数告诉我们风险范围。

In [ ]:
def distribution_summary(series):
    series = pd.Series(series)
    return pd.Series({
        'Mean': series.mean(),
        'Std Dev': series.std(ddof=1),
        'P5': series.quantile(0.05),
        'Median': series.median(),
        'P95': series.quantile(0.95),
        'Minimum': series.min(),
        'Maximum': series.max(),
    })

summary_table = pd.DataFrame({
    'Basis ($/bu)': distribution_summary(basis),
    'Harvest Futures ($/bu)': distribution_summary(harvest_futures),
    'Cash Price ($/bu)': distribution_summary(cash_price),
}).T

print(summary_table.round(4).to_string())

## 12. 画出Basis和Cash Price分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(basis, bins=35, color='#ED7D31', edgecolor='white')
axes[0].axvline(BASIS_MODE, color='black', linestyle='--', label='Mode = -0.20')
axes[0].set_title('Simulated Iowa Harvest Basis')
axes[0].set_xlabel('USD per bushel')
axes[0].set_ylabel('Number of simulations')
axes[0].legend()

axes[1].hist(cash_price, bins=35, color='#70AD47', edgecolor='white')
axes[1].axvline(cash_price.mean(), color='black', linestyle='--', label='Mean')
axes[1].set_title('Simulated Harvest Cash Price')
axes[1].set_xlabel('USD per bushel')
axes[1].set_ylabel('Number of simulations')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. 检查Basis与Futures的关系

本项目把basis draw与期货价格过程独立抽取，所以二者相关系数应接近0。Cash price则主要由Harvest futures驱动，因此二者相关系数应非常接近1。

这是一个透明的简化假设；它不代表现实中的basis与futures永远完全独立。

In [ ]:
relationship_checks = pd.Series({
    'Corr(Basis, Harvest Futures)': np.corrcoef(basis, harvest_futures)[0, 1],
    'Corr(Cash Price, Harvest Futures)': np.corrcoef(cash_price, harvest_futures)[0, 1],
    'Corr(Cash Price, Basis)': np.corrcoef(cash_price, basis)[0, 1],
})

print(relationship_checks.round(6).to_string())

## 14. 可重复性检查

如果没有修改seed、公式或输入文件，下面的结果应完全通过。

In [ ]:
expected = {
    'basis_mean': -0.19706122942428791,
    'basis_std': 0.034991599725882054,
    'cash_mean': 4.344238216666843,
    'cash_std': 0.7845871974653958,
    'cash_p5': 2.8990342167936682,
    'cash_p95': 5.7534050386928195,
}

actual = {
    'basis_mean': basis.mean(),
    'basis_std': basis.std(ddof=1),
    'cash_mean': cash_price.mean(),
    'cash_std': cash_price.std(ddof=1),
    'cash_p5': np.quantile(cash_price, 0.05),
    'cash_p95': np.quantile(cash_price, 0.95),
}

for key in expected:
    assert np.isclose(actual[key], expected[key], atol=1e-12), (key, actual[key], expected[key])

print('可重复性检查通过。')
print(pd.DataFrame({'Expected': expected, 'Actual': actual}).round(6).to_string())

## 15. 保存第6课结果

完整情景表会保存到`lesson_06_outputs`。下一课可以直接使用它计算production、cash revenue和production cost。

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_06_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

scenario_path = OUTPUT_DIR / 'cash_price_scenarios_10000.csv'
summary_path = OUTPUT_DIR / 'step_06_summary.json'

scenarios.to_csv(scenario_path, index=False)

summary_for_json = {
    'n_simulations': N_SIMULATIONS,
    'random_seed': RANDOM_SEED,
    'basis_source': 'Iowa State Ag Decision Maker A2-41, Iowa state average, November week 1',
    'basis_definition': 'cash price minus December futures price',
    'basis_triangular_parameters_usd_per_bushel': {
        'low': BASIS_LOW,
        'mode': BASIS_MODE,
        'high': BASIS_HIGH,
    },
    'cash_price_floor_usd_per_bushel': CASH_PRICE_FLOOR,
    'basis_summary': {k: float(v) for k, v in distribution_summary(basis).items()},
    'cash_price_summary': {k: float(v) for k, v in distribution_summary(cash_price).items()},
}

summary_path.write_text(json.dumps(summary_for_json, indent=2), encoding='utf-8')

print('已保存:', scenario_path)
print('已保存:', summary_path)

## 16. 本课结论与限制

现在每个未来情景已经同时包含：产量、July futures、Harvest futures、basis和cash price。

本课必须保留的说明：

1. `Basis = Cash Price − Futures Price`，所以负basis会让cash price低于futures；
2. 三角分布参数来自Iowa State州平均、November week 1、December contract的五年范围与平均值；
3. 把五年平均`−0.20`当作三角分布众数，是建模假设；
4. Basis与期货价格独立抽取，是简化假设；
5. $0.50/bu cash-price floor是模型假设，本次10,000次模拟中没有触发；
6. 还没有加入套保，所以现在只能说“现金卖粮价格是多少”，还不能判断哪种策略最好。

**下一课：计算Production、Cash Revenue、Production Cost和Unhedged Profit，建立后续所有套保策略的比较基准。**